# H2O：让高累计注意力 Token 留在有限 KV Cache

**面试问题：Heavy-Hitter Oracle 如何在滑动窗口之外保留长期重要 Token？**

## 回答主线

1. 固定容量 KV Cache 的目标不是保存最近 Token 本身，而是最大化后续查询仍能访问的注意力质量。
2. 纯滑动窗口对局部连贯有利，却会驱逐开头的政策、身份和约束 Token。
3. H2O 维护历史累计注意力分数，将容量拆成 heavy-hitter 区和最近窗口区。
4. 每次驱逐必须同时更新物理块表与逻辑位置，不能只保留一个索引集合。
5. 累计分数也会被早期尖峰污染，因此长会话需要衰减或分阶段重估。
6. 评估应报告保留注意力质量、关键字段召回和真实延迟，而不只报告缓存命中率。

## 真实案例

一个客服对话有十二个上下文 Token，其中开头包含退款政策和订单号，末尾是最近追问。六个后续解码步形成可读的注意力权重；缓存只能保留六个 Token。我们比较最近六个的滑动窗口、累计注意力 heavy hitter 加最近窗口，以及带衰减的修正。教学实验使用可读的小数据解释机制，结果不能外推为线上收益。

### 输入预览：上下文 Token 与六步注意力

In [1]:
import numpy as np  # 导入数组运算以计算注意力质量和缓存策略。

tokens = ["系统", "退款3天", "用户", "订单A7", "已经", "付款", "客服", "核验", "用户", "再次", "追问", "进度"]  # 构造含长期政策和近期对话的十二个 Token。
attention = np.array([  # 构造六个后续解码步对上下文的完整注意力分布。
    [0.03, 0.25, 0.02, 0.18, 0.03, 0.03, 0.04, 0.08, 0.04, 0.06, 0.10, 0.14],  # 第一步同时依赖政策、订单和最近问题。
    [0.02, 0.22, 0.02, 0.20, 0.03, 0.03, 0.03, 0.07, 0.04, 0.07, 0.12, 0.15],  # 第二步继续引用订单实体。
    [0.02, 0.18, 0.02, 0.22, 0.02, 0.03, 0.03, 0.06, 0.05, 0.08, 0.13, 0.16],  # 第三步提高近期上下文权重。
    [0.02, 0.24, 0.02, 0.16, 0.02, 0.02, 0.03, 0.05, 0.05, 0.08, 0.14, 0.17],  # 第四步生成政策天数时重新关注开头。
    [0.02, 0.20, 0.02, 0.15, 0.02, 0.02, 0.03, 0.05, 0.06, 0.09, 0.15, 0.19],  # 第五步平衡政策和最近问题。
    [0.02, 0.17, 0.02, 0.14, 0.02, 0.02, 0.03, 0.05, 0.07, 0.10, 0.16, 0.20],  # 第六步更依赖最近 Token。
], dtype=float)  # 完成注意力矩阵。
attention = attention / attention.sum(axis=1, keepdims=True)  # 对每个解码步归一化以便解释为概率质量。
print(f"上下文长度={len(tokens)}，解码步数={attention.shape[0]}，缓存容量=6")  # 输出实验形状。
print("位置  Token       首步注意力  六步累计")  # 输出输入统计表头。
for index, token in enumerate(tokens):  # 逐 Token 展示其长期重要性。
    print(f"{index:>2}   {token:<9} {attention[0, index]:>9.3f} {attention[:, index].sum():>9.3f}")  # 展示政策与订单的累计权重。

上下文长度=12，解码步数=6，缓存容量=6
位置  Token       首步注意力  六步累计
 0   系统            0.030     0.130
 1   退款3天          0.250     1.260
 2   用户            0.020     0.120
 3   订单A7          0.180     1.050
 4   已经            0.030     0.140
 5   付款            0.030     0.150
 6   客服            0.040     0.190
 7   核验            0.080     0.360
 8   用户            0.040     0.310
 9   再次            0.060     0.480
10   追问            0.100     0.800
11   进度            0.140     1.010


## Baseline 基线：只保留最近六个 Token

In [2]:
capacity = 6  # 固定物理 KV Cache 容量。
sliding_indices = np.arange(len(tokens) - capacity, len(tokens))  # 选择最近六个 Token 作为滑动窗口。
sliding_mass = attention[:, sliding_indices].sum(axis=1)  # 计算每个解码步仍可访问的完整注意力质量。
critical_indices = {tokens.index("退款3天"), tokens.index("订单A7")}  # 定义回答正确必须保留的政策和订单位置。
sliding_critical_recall = len(critical_indices.intersection(set(sliding_indices))) / len(critical_indices)  # 计算关键字段召回率。
print("滑动窗口保留：", [tokens[index] for index in sliding_indices])  # 展示最近窗口驱逐了哪些长期 Token。
print("逐步保留注意力质量：", np.round(sliding_mass, 3).tolist())  # 展示每个生成步可见的权重质量。
print(f"平均质量={sliding_mass.mean():.3f}，关键字段召回={sliding_critical_recall:.0%}")  # 输出可比较基线指标。

滑动窗口保留： ['客服', '核验', '用户', '再次', '追问', '进度']
逐步保留注意力质量： [0.46, 0.48, 0.51, 0.52, 0.57, 0.61]
平均质量=0.525，关键字段召回=0%


### 核心实现：累计 Heavy Hitter 与最近窗口配额

In [3]:
def h2o_select(history_attention, total_capacity, recent_budget):  # 实现累计注意力 heavy hitter 加最近窗口的选择器。
    recent = np.arange(len(tokens) - recent_budget, len(tokens))  # 固定保留最近 Token 维持局部连贯。
    candidate_indices = np.arange(0, len(tokens) - recent_budget)  # 仅在较早 Token 中竞争 heavy-hitter 配额。
    cumulative_scores = history_attention.sum(axis=0)  # 累积已观察解码步对每个 KV 的注意力贡献。
    heavy_budget = total_capacity - recent_budget  # 计算可供长期重要 Token 使用的容量。
    ranked = candidate_indices[np.argsort(-cumulative_scores[candidate_indices], kind="stable")]  # 按累计分数稳定降序排列早期 Token。
    heavy = ranked[:heavy_budget]  # 选择得分最高的长期 Token。
    selected = np.sort(np.concatenate([heavy, recent]))  # 合并并恢复逻辑位置顺序。
    return selected, cumulative_scores  # 返回缓存索引和可观测累计分数。

calibration_attention = attention[:3]  # 使用前三个已发生解码步更新 H2O 分数。
h2o_indices, cumulative_scores = h2o_select(calibration_attention, total_capacity=capacity, recent_budget=2)  # 分配四个 heavy 和两个 recent 槽位。
h2o_mass = attention[3:, h2o_indices].sum(axis=1)  # 在后三个未来步骤评估保留注意力质量。
h2o_critical_recall = len(critical_indices.intersection(set(h2o_indices))) / len(critical_indices)  # 计算关键字段召回率。
print("累计分数 Top 排名：")  # 输出 heavy-hitter 选择依据。
for index in np.argsort(-cumulative_scores)[:8]:  # 展示最高分的八个 Token。
    print(f"token={tokens[index]:<9} score={cumulative_scores[index]:.3f} selected={index in set(h2o_indices)}")  # 显示长期政策和订单是否进入缓存。
print("H2O 保留：", [tokens[index] for index in h2o_indices])  # 输出最终逻辑缓存内容。

累计分数 Top 排名：
token=退款3天      score=0.650 selected=True
token=订单A7      score=0.600 selected=True
token=进度        score=0.450 selected=True
token=追问        score=0.350 selected=True
token=核验        score=0.210 selected=True
token=再次        score=0.210 selected=True
token=用户        score=0.130 selected=False
token=客服        score=0.100 selected=False
H2O 保留： ['退款3天', '订单A7', '核验', '再次', '追问', '进度']


## 结果解读：保留质量、关键召回与位置账本

In [4]:
future_sliding_mass = sliding_mass[3:]  # 取与 H2O 相同的后三步评估窗口。
print("方案          step4  step5  step6  平均质量  关键召回")  # 输出同口径结果表头。
print(f"Sliding      {future_sliding_mass[0]:.3f}  {future_sliding_mass[1]:.3f}  {future_sliding_mass[2]:.3f}    {future_sliding_mass.mean():.3f}      {sliding_critical_recall:.0%}")  # 展示滑动窗口结果。
print(f"H2O          {h2o_mass[0]:.3f}  {h2o_mass[1]:.3f}  {h2o_mass[2]:.3f}    {h2o_mass.mean():.3f}      {h2o_critical_recall:.0%}")  # 展示 H2O 结果。
logical_to_physical = {int(logical): slot for slot, logical in enumerate(h2o_indices)}  # 构造逻辑位置到连续物理槽位的块表。
print("逻辑位置 -> 物理槽：", logical_to_physical)  # 展示实现不能只保存 Token 字符串。
print("解读：H2O 用四个长期槽保住政策和订单，两个最近槽保住当前问题；收益来自未来确实仍关注这些 Token。")  # 解释质量提升原因。

方案          step4  step5  step6  平均质量  关键召回
Sliding      0.520  0.570  0.610    0.567      0%
H2O          0.840  0.830  0.820    0.830      100%
逻辑位置 -> 物理槽： {1: 0, 3: 1, 7: 2, 9: 3, 10: 4, 11: 5}
解读：H2O 用四个长期槽保住政策和订单，两个最近槽保住当前问题；收益来自未来确实仍关注这些 Token。


## 失败案例：早期注意力尖峰永久霸占缓存

In [5]:
spiky_history = calibration_attention.copy()  # 复制校准注意力以构造早期异常尖峰。
spiky_history[0, tokens.index("付款")] += 1.20  # 让一次异常查询对普通 Token 产生极高注意力。
plain_indices, plain_scores = h2o_select(spiky_history, total_capacity=capacity, recent_budget=2)  # 使用无衰减累计分数选择缓存。
decay_weights = np.array([0.25, 0.50, 1.00])[:, None]  # 给越新的解码步越高权重以遗忘陈旧尖峰。
decayed_indices, decayed_scores = h2o_select(spiky_history * decay_weights, total_capacity=capacity, recent_budget=2)  # 使用指数式时间衰减重新选择。
plain_future_mass = attention[3:, plain_indices].sum(axis=1).mean()  # 评估尖峰污染后的未来质量。
decayed_future_mass = attention[3:, decayed_indices].sum(axis=1).mean()  # 评估衰减修正后的未来质量。
print("无衰减保留：", [tokens[index] for index in plain_indices], f"未来质量={plain_future_mass:.3f}")  # 展示陈旧付款 Token 占槽。
print("衰减后保留：", [tokens[index] for index in decayed_indices], f"未来质量={decayed_future_mass:.3f}")  # 展示近期证据恢复合理排序。
print("修正策略：累计分数需要时间衰减、层/头归一化和最小 recent 配额，不能把一次尖峰当永久重要性。")  # 总结边界条件。

无衰减保留： ['退款3天', '订单A7', '付款', '核验', '追问', '进度'] 未来质量=0.760
衰减后保留： ['退款3天', '订单A7', '付款', '再次', '追问', '进度'] 未来质量=0.800
修正策略：累计分数需要时间衰减、层/头归一化和最小 recent 配额，不能把一次尖峰当永久重要性。


### 生产边界与驱逐事件

In [6]:
evicted = [tokens[index] for index in range(len(tokens)) if index not in set(h2o_indices)]  # 计算被 H2O 驱逐的 Token 以形成审计事件。
cache_event = {"capacity": capacity, "heavy_budget": 4, "recent_budget": 2, "kept": [tokens[index] for index in h2o_indices], "evicted": evicted, "score_version": "attn-sum-r1"}  # 构造缓存决策账本。
print("缓存事件：", cache_event)  # 展示线上排查回答退化所需字段。
print("生产替换点：真实实现还需逐层逐头统计、Paged KV 物理释放、Prefill/Decode 分离、量化缓存和 Kernel 延迟测量。")  # 明确 NumPy 索引与真实 Serving 的差距。

缓存事件： {'capacity': 6, 'heavy_budget': 4, 'recent_budget': 2, 'kept': ['退款3天', '订单A7', '核验', '再次', '追问', '进度'], 'evicted': ['系统', '用户', '已经', '付款', '客服', '用户'], 'score_version': 'attn-sum-r1'}
生产替换点：真实实现还需逐层逐头统计、Paged KV 物理释放、Prefill/Decode 分离、量化缓存和 Kernel 延迟测量。


## 回归测试：最后只保护容量、关键召回与失败修正

In [7]:
assert len(h2o_indices) == capacity and len(set(h2o_indices)) == capacity  # 验证缓存严格满足固定容量且没有重复槽位。
assert h2o_critical_recall == 1.0 and sliding_critical_recall == 0.0  # 验证 H2O 保住政策与订单而滑动窗口全部丢失。
assert h2o_mass.mean() > future_sliding_mass.mean()  # 验证同一未来注意力下 H2O 保留质量更高。
assert tokens.index("付款") in set(plain_indices)  # 验证早期尖峰污染反例确实发生。
assert decayed_future_mass > plain_future_mass  # 验证时间衰减改善未来保留质量。
print("回归测试通过：固定容量、长期关键召回、注意力质量、尖峰失败和衰减修正均成立。")  # 用少量断言总结缓存合同。

回归测试通过：固定容量、长期关键召回、注意力质量、尖峰失败和衰减修正均成立。
